# IoT Predictive Maintenance — Phase 1: Model Training & Evaluation

This notebook covers the full training pipeline, from the **Randomized Shuffle Split** to the final **Isolation Forest** model.

## 1. The 3-Way Split Strategy
We partitioned the data into three silos to ensure clean benchmarking and pipeline testing:
1. **Training (80% of Half A)**
2. **Benchmark (20% of Half A)**
3. **Pipeline Test (Half B)** - Completely unseen reserved for Salesforce testing.

In [ ]:
import os
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

DATA_DIR = r'D:\IOT Project\predictive-maintenance-ai\data'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_80_of_half_a.csv'))
bench_df = pd.read_csv(os.path.join(DATA_DIR, 'benchmark_20_of_half_a.csv'))

sensor_cols = [c for c in train_df.columns if c not in ['timestamp', 'original_index']]

print(f"Training rows: {len(train_df)}")
print(f"Benchmark rows: {len(bench_df)}")

## 2. Pipeline Training
We use a `Pipeline` to ensure that the **StandardScaler** (normalization) is saved along with the **IsolationForest** model. This prevents 'Normalization Leakage' and ensures consistent inference.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', IsolationForest(contamination=0.05, random_state=42))
])

pipeline.fit(train_df[sensor_cols])
print("✅ Pipeline trained.")

## 3. Heuristic Validation
Since the data is unlabeled, we validate the model by checking if it correctly identifies high-vibration energy samples as anomalies.

In [ ]:
preds = pipeline.predict(bench_df[sensor_cols])
bench_df['is_anomaly'] = (preds == -1)

summary = bench_df.groupby('is_anomaly')[['fwd_rms_vel', 'rr_rms_vel', 'fwd_kurtosis']].mean()
print("Comparison of Normal vs Anomalous samples:")
summary

## 4. Saving the 'Brain'
The model is saved as a `.joblib` file for use in the live Orchestrator.

In [ ]:
MODEL_PATH = r'D:\IOT Project\predictive-maintenance-ai\models\fsl_heavy_motor_model.joblib'
joblib.dump(pipeline, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")